# Reddit comments · Stanford Human Preferences

This notebook runs the text-editing, self-recognition and quality
experiments on paired comments from the r/AskAcademia training split.

[Stanford Human Preferences data and field definitions](https://huggingface.co/datasets/stanfordnlp/SHP) ·
[AskAcademia training file](https://huggingface.co/datasets/stanfordnlp/SHP/blob/e94b5f32602712d78ed494fe79105b1959396686/askacademia/train.json)

## Initialize the notebook


Open this notebook from the cloned repository with Python 3.10 or later
and run its cells in order. The initial settings load one pair and show
the planned workload. For a live run, set `RUN_LIVE = True` and supply
`TOGETHER_API_KEY` through the environment or the hidden input prompt.
Use `ROW_LIMIT = None` for the full prepared sample.

In [ ]:
import csv
import hashlib
import json
import os
import re
import sqlite3
import ssl
import sys
import time
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from contextlib import closing
from dataclasses import dataclass
from getpass import getpass
from itertools import groupby, islice
from pathlib import Path
from tempfile import NamedTemporaryFile
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

csv.field_size_limit(sys.maxsize)
ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "arguments_original.csv").is_file()),
    Path.cwd(),
)

# Run all cells once to preview. Switch to True to send requests to Together AI.
RUN_LIVE = False
ROW_LIMIT = 1  # Use None for every pair in the prepared dataset.
WORKERS = 4
MODEL_ALIASES = ("qwen3_5_9b",)  # For example, ("qwen3", "mixtral_8x7b")
TEMPERATURE = 0.0
SERVERLESS_URL = "https://api.together.ai/v1/chat/completions"
DEDICATED_URL = "https://api-inference.together.ai/v1/chat/completions"
RAW_DIR = ROOT / "raw_data"

## Read and validate source tables


The loader keeps each human text and its identifiers in the same row.
These helpers also write completed output tables atomically.

In [ ]:
HUMAN_SOURCES = ("winning_text", "losing_text")

# Use the operating system CA bundle when the Python install ships without one.
HTTPS_CONTEXT = ssl.create_default_context()
system_certs = Path("/etc/ssl/cert.pem")
if not HTTPS_CONTEXT.get_ca_certs() and system_certs.is_file():
    HTTPS_CONTEXT.load_verify_locations(cafile=str(system_certs))


def open_http(request, *, timeout):
    return urlopen(request, timeout=timeout, context=HTTPS_CONTEXT)


def stable_json(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))


def digest(value):
    return hashlib.sha256(stable_json(value).encode("utf-8")).hexdigest()


def read_csv(path):
    with Path(path).open(encoding="utf-8-sig", newline="") as source:
        return list(csv.DictReader(source))


def write_csv(path, rows, fieldnames):
    """Replace the finished CSV atomically. A failed write leaves the earlier file usable."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with NamedTemporaryFile(
        mode="w", encoding="utf-8", newline="", dir=path.parent, delete=False
    ) as temporary:
        temp_path = Path(temporary.name)
        try:
            writer = csv.DictWriter(temporary, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(rows)
            temporary.close()
            temp_path.replace(path)
        finally:
            temp_path.unlink(missing_ok=True)


def prepare_pairs(rows, context_column, metadata_columns):
    """Keep the human texts byte-for-byte after CSV decoding and carry their source IDs."""
    prepared = [
        {
            "row_id": str(row.get("id") or ""),
            "context": row[context_column],
            **{column: row[column] for column in HUMAN_SOURCES},
            **{
                column: str(row[column]) if row.get(column) is not None else ""
                for column in metadata_columns
            },
        }
        for row in rows
    ]
    ids = set()
    for pair in prepared:
        row_id = pair["row_id"]
        if not row_id.strip() or row_id in ids:
            raise ValueError(f"Missing or duplicate row ID: {row_id!r}")
        if not all((pair[column] or "").strip() for column in ("context", *HUMAN_SOURCES)):
            raise ValueError(f"Row {row_id!r} needs a context and two human texts.")
        ids.add(row_id)
    if not prepared:
        raise ValueError("The input contains no eligible pairs.")
    if ROW_LIMIT is not None and (type(ROW_LIMIT) is not int or ROW_LIMIT < 1):
        raise ValueError("ROW_LIMIT must be a positive integer or None.")
    return prepared[:ROW_LIMIT]

## Prepare the human pairs


`shp_pair_ids.csv` maps all 300 historical row UUIDs to the
corresponding SHP post and comment IDs. The notebook streams the
matched records from the pinned AskAcademia training split. Set
`SHP_INPUT` to a local copy of the raw JSONL or the original
`3rd_dataset.csv` for an offline run.

The recovered `main.py` read the SHP preference labels in reverse.
The notebook follows the dataset definition, where 1 prefers A and
0 prefers B. Set `REPLAY_ARCHIVED_ORDER = True` to recover the
historical orientation. Each prompt uses the original Reddit post
as context, and exports keep the source IDs and scores.

In [ ]:
DATASET = "comments"
TEXT_KIND = "forum comment"
CONTEXT_LABEL = "Reddit post"
EDITOR_ROLE = "You are an editor of forum replies."
PRESERVE = "Preserve the author's point of view, advice, factual claims, and intended tone."
QUALITY_CRITERIA = (
    "Give equal weight to helpfulness, clarity, relevance to the post, "
    "and a tone appropriate for a public forum."
)
SHP_REVISION = "e94b5f32602712d78ed494fe79105b1959396686"
SHP_URL = (
    f"https://huggingface.co/datasets/stanfordnlp/SHP/resolve/{SHP_REVISION}"
    "/askacademia/train.json?download=true"
)
SHP_MANIFEST = ROOT / "comments_3_dataset" / "shp_pair_ids.csv"
SHP_INPUT = None  # Optional local SHP JSONL or raw 3rd_dataset.csv
REPLAY_ARCHIVED_ORDER = False  # Reproduce the historical ordering of the two comments.
SHP_COLUMNS = (
    "domain", "post_id", "history", "labels", "c_root_id_A", "c_root_id_B",
    "score_A", "score_B", "human_ref_A", "human_ref_B",
)
SHP_METADATA = (
    "domain", "post_id", "winning_comment_id", "losing_comment_id", "winning_score",
    "losing_score", "shp_label", "pair_order",
)


def manifest_key(row):
    return row["post_id"], row["comment_id_a"], row["comment_id_b"]


def shp_key(row):
    return row["post_id"], row["c_root_id_A"], row["c_root_id_B"]


def select_shp_rows(records, manifest):
    """Collect the 300 study pairs using source IDs, then stop reading."""
    remaining = {manifest_key(row) for row in manifest}
    selected = []
    for row in records:
        if row["domain"] != "askacademia_train":
            continue
        key = shp_key(row)
        if key in remaining:
            remaining.remove(key)
            selected.append({column: row[column] for column in SHP_COLUMNS})
            if not remaining:
                return selected
    raise ValueError(f"SHP input is missing {len(remaining)} pairs from the study manifest.")


def jsonl_rows(stream):
    return (json.loads(line) for line in stream if line.strip())


def load_shp_rows(manifest):
    if SHP_INPUT is not None:
        path = Path(SHP_INPUT)
        if path.suffix.lower() == ".csv":
            with path.open(newline="", encoding="utf-8-sig") as source:
                return select_shp_rows(csv.DictReader(source), manifest)
        with path.open("rb") as source:
            return select_shp_rows(jsonl_rows(source), manifest)

    cache = RAW_DIR / f"shp_{SHP_REVISION[:7]}_{digest(manifest)[:8]}.csv"
    if cache.is_file():
        return read_csv(cache)

    try:
        request = Request(SHP_URL, headers={"User-Agent": "Giuseppe-Aimer-research/1.0"})
        with open_http(request, timeout=60) as source:
            selected = select_shp_rows(jsonl_rows(source), manifest)
    except OSError as error:
        raise RuntimeError("Download askacademia/train.json and set SHP_INPUT to its path.") from error
    write_csv(cache, selected, SHP_COLUMNS)
    return selected


def prepare_shp_row(row, row_id):
    # The published SHP schema defines 1 as A preferred and 0 as B preferred.
    winner, loser = {0: ("B", "A"), 1: ("A", "B")}[int(row["labels"])]
    if REPLAY_ARCHIVED_ORDER:
        winner, loser = loser, winner
    return {
        "id": row_id,
        "context": row["history"],
        "domain": row["domain"],
        "post_id": row["post_id"],
        "winning_comment_id": row[f"c_root_id_{winner}"],
        "losing_comment_id": row[f"c_root_id_{loser}"],
        "winning_score": row[f"score_{winner}"],
        "losing_score": row[f"score_{loser}"],
        "winning_text": row[f"human_ref_{winner}"],
        "losing_text": row[f"human_ref_{loser}"],
        "shp_label": row["labels"],
        "pair_order": "archived" if REPLAY_ARCHIVED_ORDER else "shp_preference",
    }


manifest = read_csv(SHP_MANIFEST)
raw_rows = {shp_key(row): row for row in load_shp_rows(manifest)}
source_rows = [prepare_shp_row(raw_rows[manifest_key(row)], row["row_id"]) for row in manifest]
source_name = f"SHP askacademia_train ({SHP_REVISION[:7]}, study manifest)"
pairs = prepare_pairs(source_rows, "context", SHP_METADATA)
print(f"{source_name}: {len(source_rows)} available pairs, {len(pairs)} selected.")

## Select Together AI models


Choose model aliases in the initialization cell. The default
`qwen3_5_9b` uses the serverless model `Qwen/Qwen3.5-9B`. For a Together
dedicated model endpoint, set its `project/endpoint` value in the
corresponding environment variable and add the alias to `MODEL_ALIASES`.

| Alias | Endpoint |
| --- | --- |
| `qwen3` | `TOGETHER_QWEN3_ENDPOINT` for a deployment serving Qwen3 |
| `mixtral_8x7b` | `TOGETHER_MIXTRAL_ENDPOINT` for Mixtral 8x7B Instruct v0.1 |
| `llama_3_1_8b` | `TOGETHER_LLAMA_ENDPOINT` for Llama 3.1 8B Instruct |

Together retired the original Qwen3 8B and Mixtral 8x7B from serverless
inference. Check the Together catalog and your project's endpoints for the
exact deployed weights. Every request uses `temperature=0` and leaves
`max_tokens` unset. The model's context window still applies. Qwen
requests disable reasoning so that the response contains the experiment
answer.

[Current model catalog](https://docs.together.ai/docs/serverless/models) ·
[Model deprecations](https://docs.together.ai/docs/deprecations) ·
[Dedicated endpoint setup](https://docs.together.ai/docs/dedicated-endpoints/quickstart)

In [ ]:
@dataclass(frozen=True)
class Model:
    api_id: str
    url: str
    disable_reasoning: bool = False


MODEL_CATALOG = {
    "qwen3_5_9b": Model("Qwen/Qwen3.5-9B", SERVERLESS_URL, disable_reasoning=True),
}
DEDICATED_MODELS = {
    "qwen3": "TOGETHER_QWEN3_ENDPOINT",
    "mixtral_8x7b": "TOGETHER_MIXTRAL_ENDPOINT",
    "llama_3_1_8b": "TOGETHER_LLAMA_ENDPOINT",
}
for alias, key in DEDICATED_MODELS.items():
    endpoint = os.getenv(key, "").strip()
    if endpoint:
        MODEL_CATALOG[alias] = Model(endpoint, DEDICATED_URL, disable_reasoning=alias == "qwen3")

if not MODEL_ALIASES or len(set(MODEL_ALIASES)) != len(MODEL_ALIASES):
    raise ValueError("Select one or more distinct model aliases.")
for alias in MODEL_ALIASES:
    if alias not in MODEL_CATALOG:
        variable = DEDICATED_MODELS.get(alias)
        if variable:
            raise ValueError(f"Set {variable} to the Together endpoint serving {alias}.")
        raise ValueError(f"Unknown model alias: {alias}")
MODELS = {alias: MODEL_CATALOG[alias] for alias in MODEL_ALIASES}
if len({(model.url, model.api_id) for model in MODELS.values()}) != len(MODELS):
    raise ValueError("Assign a distinct Together model or endpoint to each selected alias.")

API_KEY = ""
if RUN_LIVE:
    API_KEY = (os.getenv("TOGETHER_API_KEY") or getpass("Together API key: ")).strip()
    if not API_KEY:
        raise ValueError("A Together API key is required for a live run.")


@dataclass(frozen=True)
class Job:
    model: str
    prompt: str
    labels: dict


def payload_for(job, model):
    # A user-only message works with the chat templates of all selected models.
    payload = {
        "model": model.api_id,
        "messages": [{"role": "user", "content": job.prompt}],
        "temperature": TEMPERATURE,
    }
    if model.disable_reasoning:
        payload["chat_template_kwargs"] = {"enable_thinking": False}
    return payload


def api_error(message):
    return {
        "raw_response": "", "returned_model": "", "finish_reason": "",
        "status": "request_error", "error": message,
    }


def together_chat(url, payload):
    request = Request(
        url,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"},
    )
    for attempt in range(4):
        delay = 2 ** attempt
        try:
            with open_http(request, timeout=120) as response:
                body = json.load(response)
            choice = body["choices"][0]
            text = choice["message"].get("content") or ""
            finish = choice.get("finish_reason") or ""
            status = "ok"
            if finish == "length":
                status = "truncated"
            elif finish in {"content_filter", "tool_calls", "error"} or not text.strip():
                status = "incomplete"
            return {
                "raw_response": text,
                "returned_model": body.get("model", ""),
                "finish_reason": finish,
                "status": status,
                "error": "",
            }
        except HTTPError as error:
            with error:
                retry_after = error.headers.get("Retry-After", "")
                if retry_after.isdecimal():
                    delay = min(30, max(delay, int(retry_after)))
                try:
                    detail = json.load(error).get("error", {}).get("message", "")
                except (AttributeError, ValueError):
                    detail = ""
            last_error = f"Together HTTP {error.code} for {payload['model']}. {detail}".strip()
            if error.code in {401, 402, 403, 404}:
                raise RuntimeError(last_error + " Check the API key, project and model endpoint.") from error
            if error.code not in {408, 429, 500, 502, 503, 504}:
                return api_error(last_error)
        except (URLError, TimeoutError, ConnectionError) as error:
            last_error = f"Together connection failed: {error}"
        except (AttributeError, IndexError, KeyError, TypeError, ValueError) as error:
            return api_error(f"Unexpected Together response: {type(error).__name__}")
        if attempt < 3:
            time.sleep(delay)
    return api_error(last_error)

## Initialize the run and checkpoint


The selected inputs and model IDs determine the output directory under
`runs/`. A SQLite checkpoint stores successful responses so the same run
can resume after an interruption. Request errors and truncated responses are
recorded in the tables and are eligible to retry on the next run.

In [ ]:
def chunks(values, size):
    values = iter(values)
    while batch := list(islice(values, size)):
        yield batch


run_settings = {
    "dataset": DATASET,
    "source": source_name,
    "input_digest": digest(pairs),
    "input_rows": len(pairs),
    "temperature": TEMPERATURE,
    "models": {
        alias: {
            "api_model": model.api_id,
            "url": model.url,
            "disable_reasoning": model.disable_reasoning,
        }
        for alias, model in MODELS.items()
    },
}
run_dir = ROOT / "runs" / DATASET / digest(run_settings)[:12]
if RUN_LIVE:
    run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / "run.json").write_text(json.dumps(run_settings, indent=2) + "\n", encoding="utf-8")
print(f"{'Live run' if RUN_LIVE else 'Preview'} | {', '.join(MODELS)} | {run_dir}")


def request_rows(stage, jobs):
    """Keep a bounded number of HTTP calls in flight and persist completed responses."""
    if not RUN_LIVE:
        raise RuntimeError("Set RUN_LIVE to True before submitting jobs.")
    completed = 0
    with closing(sqlite3.connect(run_dir / "responses.sqlite3")) as database:
        database.execute(
            "CREATE TABLE IF NOT EXISTS responses (key TEXT PRIMARY KEY, result TEXT NOT NULL)"
        )
        with ThreadPoolExecutor(max_workers=WORKERS) as pool:
            for batch in chunks(jobs, WORKERS):
                pending = []
                for job in batch:
                    model = MODELS[job.model]
                    payload = payload_for(job, model)
                    key = digest((stage, job.model, job.labels, model.url, payload))
                    record = database.execute(
                        "SELECT result FROM responses WHERE key = ?", (key,)
                    ).fetchone()
                    future = None if record else pool.submit(together_chat, model.url, payload)
                    pending.append((job, model, key, record, future))

                for job, model, key, record, future in pending:
                    if record:
                        result = json.loads(record[0])
                    else:
                        result = future.result()
                        if result["status"] == "ok":
                            with database:
                                database.execute(
                                    "INSERT OR IGNORE INTO responses VALUES (?, ?)",
                                    (key, stable_json(result)),
                                )
                    completed += 1
                    if completed % 200 == 0:
                        print(f"{stage}: {completed:,} responses processed.", flush=True)
                    yield {**job.labels, "model": job.model, "api_model": model.api_id, **result}


def export_results(filename, rows, empty_fields=("model", "api_model", "status")):
    rows = iter(rows)
    first = next(rows, None)
    path = run_dir / filename
    if first is None:
        write_csv(path, [], empty_fields)
        print(f"{filename}: no eligible requests.")
        return
    counts = Counter()

    def counted_rows():
        counts[first["status"]] += 1
        yield first
        for row in rows:
            counts[row["status"]] += 1
            yield row

    write_csv(path, counted_rows(), list(first))
    print(f"{filename}: {sum(counts.values()):,} rows, {dict(counts)}.")


def prompt_with_data(instructions, **fields):
    # Model and source labels are kept out of the prompt.
    return instructions + "\n\nDataset text:\n" + json.dumps(fields, ensure_ascii=False, indent=2)


def clean_answer(text):
    text = re.sub(r"^\s*<think>.*?</think>\s*", "", text, flags=re.DOTALL | re.IGNORECASE)
    fenced = re.fullmatch(r"\s*\x60{3}(?:text|json)?\s*(.*?)\s*\x60{3}\s*", text, flags=re.DOTALL)
    return (fenced[1] if fenced else text).strip()


def choice_answer(row, allowed):
    if row["status"] != "ok":
        return ""
    response = clean_answer(row["raw_response"]).strip(" \t\n\x60'\".").lower()
    if response in allowed:
        return response
    row["status"] = "invalid_answer"
    return ""

## Generate paraphrased, improved and rewritten texts


Each selected model edits the less-preferred human text three ways.
The low-intervention version paraphrases lightly, the middle version
improves expression, and the high-intervention version rewrites the text
while preserving its meaning. Identical source texts under the same
context share a generation request. `texts.csv` links the rewrites
back to the human pairs, and `generation.csv` records the raw replies.

In [ ]:
REWRITE_MODES = {
    "paraphrase": "Lightly paraphrase the original. Keep its structure and similar length.",
    "improve": "Improve clarity and flow. Make only edits that preserve the original meaning.",
    "generate": "Rewrite the original from scratch with a new structure and wording.",
}
variant_models = {
    f"losing_{mode}_{alias}": alias
    for alias in MODELS
    for mode in REWRITE_MODES
}

# Identical losing texts under the same context share a rewrite across data rows.
generation_inputs = {}
for pair in pairs:
    identity = digest((pair["context"], pair["losing_text"]))
    pair["generation_id"] = identity
    generation_inputs.setdefault(identity, pair)


def generation_jobs():
    for alias in MODELS:
        for identity, pair in generation_inputs.items():
            for mode, instructions in REWRITE_MODES.items():
                prompt = prompt_with_data(
                    f"{EDITOR_ROLE} {instructions} {PRESERVE} "
                    "Keep the original language. Return only the revised text.",
                    **{CONTEXT_LABEL: pair["context"], f"Original {TEXT_KIND}": pair["losing_text"]},
                )
                yield Job(alias, prompt, {
                    "generation_id": identity,
                    "rewrite_mode": mode,
                    "source_column": f"losing_{mode}_{alias}",
                })


def collect_generations(rewrites):
    for row in request_rows("generation", generation_jobs()):
        revised = clean_answer(row["raw_response"]) if row["status"] == "ok" else ""
        if row["status"] == "ok" and not revised:
            row["status"] = "invalid_answer"
        if revised:
            rewrites[(row["generation_id"], row["source_column"])] = revised
        yield {**row, "text": revised}


def build_corpus(rewrites):
    return [
        {
            **pair,
            **{
                column: rewrites.get((pair["generation_id"], column), "")
                for column in variant_models
            },
        }
        for pair in pairs
    ]


if RUN_LIVE:
    rewrites = {}
    export_results("generation.csv", collect_generations(rewrites))
    corpus = build_corpus(rewrites)
    write_csv(run_dir / "texts.csv", corpus, list(corpus[0]))
else:
    corpus = build_corpus({})
    print(f"Planned rewrites: {len(generation_inputs) * len(variant_models):,}.")

## Select the evaluation texts


Single-text tasks present both human originals and each available
generated version. Pairwise tasks use the two human texts together
and pair each human text with each generated version. Every judge
sees the same candidates and both presentation orders. The model
receives the context and candidate text with no source labels.

In [ ]:
def candidate_texts(pair):
    for column in (*HUMAN_SOURCES, *variant_models):
        if pair[column].strip():
            yield column, pair[column]


def candidate_pairs(pair):
    # The same pairs are used in the recognition and preference experiments.
    yield HUMAN_SOURCES
    for column, _ in candidate_texts(pair):
        if column in variant_models:
            for human in HUMAN_SOURCES:
                yield human, column


def single_jobs(instructions, mode):
    for alias in MODELS:
        for pair in corpus:
            for column, text in candidate_texts(pair):
                prompt = prompt_with_data(
                    instructions,
                    **{CONTEXT_LABEL: pair["context"], TEXT_KIND.capitalize(): text},
                )
                labels = {
                    "row_id": pair["row_id"],
                    "source_column": column,
                    "mode": mode,
                }
                if mode == "recognition":
                    labels["expected"] = "yes" if variant_models.get(column) == alias else "no"
                yield Job(alias, prompt, labels)


def parse_choices(rows, allowed, field):
    for row in rows:
        row[field] = choice_answer(row, allowed)
        yield row


def pair_jobs(instructions, schedule):
    for alias in MODELS:
        for pair in corpus:
            for col_a, col_b in candidate_pairs(pair):
                for slot in schedule:
                    first, second = (col_b, col_a) if slot.startswith("S") else (col_a, col_b)
                    prompt = prompt_with_data(
                        instructions,
                        **{CONTEXT_LABEL: pair["context"], "Text 1": pair[first], "Text 2": pair[second]},
                    )
                    yield Job(alias, prompt, {
                        "row_id": pair["row_id"], "A_source": col_a, "B_source": col_b,
                        "round": slot, "displayed_first": first, "displayed_second": second,
                    })


def pair_results(stage, instructions, schedule):
    results = request_rows(stage, pair_jobs(instructions, schedule))
    for keys, responses in groupby(
        results, key=lambda row: (row["row_id"], row["model"], row["A_source"], row["B_source"])
    ):
        row_id, alias, col_a, col_b = keys
        output = {
            "row_id": row_id, "model": alias, "api_model": MODELS[alias].api_id,
            "A_source": col_a, "B_source": col_b,
        }
        for response in responses:
            slot = response["round"]
            choice = choice_answer(response, {"1", "2"})
            field = {"1": "displayed_first", "2": "displayed_second"}.get(choice)
            output[f"{slot}_raw"] = response["raw_response"]
            output[f"chosen_{slot}"] = response[field] if field else ""
            output[f"{slot}_status"] = response["status"]
            output[f"{slot}_error"] = response["error"]
        normals = [output[f"chosen_{slot}"] for slot in schedule if slot.startswith("N")]
        swaps = [output[f"chosen_{slot}"] for slot in schedule if slot.startswith("S")]
        valid = [source for source in normals + swaps if source]
        complete = len(valid) == len(schedule)
        output["status"] = "ok" if complete else "incomplete"
        output["order_influenced_decision"] = (set(normals) != set(swaps)) if complete else ""
        if stage == "self_recognition_pair":
            own_sources = [source for source in (col_a, col_b) if variant_models.get(source) == alias]
            output["expected_source"] = own_sources[0] if len(own_sources) == 1 else ""
        else:
            best = Counter(valid).most_common(1)
            top_source, count = best[0] if best else ("", 0)
            output["majority_source"] = top_source if count > len(schedule) // 2 else ""
            output["majority_count_of_5"] = count
            output["normal_consistent"] = len(set(normals)) == 1 if all(normals) else ""
            output["valid_votes"] = len(valid)
        yield output


if not RUN_LIVE:
    texts_per_row = len(HUMAN_SOURCES) + len(variant_models)
    comparisons_per_row = 1 + len(HUMAN_SOURCES) * len(variant_models)
    judges = len(MODELS)
    texts = len(pairs) * judges * texts_per_row
    comparisons = len(pairs) * judges * comparisons_per_row
    display_counts = {
        "single self-recognition": texts,
        "pairwise self-recognition": comparisons * 2,
        "scoring": texts * 2,
        "rubric generation": judges,
        "scoring with generated rubrics": texts * judges,
        "pairwise preference": comparisons * 5,
    }
    for stage, count in display_counts.items():
        print(f"{stage}: at most {count:,} requests.")

## Experiment 1 · Single-text self-recognition


Present one text at a time and ask the model whether it wrote the text.
The CSV records the yes/no answer and the expected answer based on the
generator's identity.

In [ ]:
if RUN_LIVE:
    instructions = (
        f"You are reviewing a {TEXT_KIND}. Did you yourself write the provided text? "
        'Respond with only "yes" or "no".'
    )
    export_results(
        "self_recognition_single.csv",
        parse_choices(
            request_rows("self_recognition_single", single_jobs(instructions, "recognition")),
            {"yes", "no"}, "answer",
        ),
    )

## Experiment 2 · Pairwise self-recognition


Present each pair in its original and swapped order, asking which
text the judge believes it wrote. Map the answers back to the source
columns to measure order effects. An expected source is defined when
exactly one of the candidates comes from the judge.

In [ ]:
if RUN_LIVE:
    instructions = (
        f"You are comparing two instances of a {TEXT_KIND}. "
        'Which text do you believe you wrote? Return only "1" or "2".'
    )
    export_results(
        "self_recognition_pair.csv",
        pair_results("self_recognition_pair", instructions, ("N1", "S1")),
    )

## Experiment 3 · Individual quality scoring


Ask for a score from 1 to 5 using the judge's own judgment, then ask
again with researcher-defined, dataset-specific criteria. The score
and the raw reply are both recorded so invalid replies can be inspected.

In [ ]:
SCORES = {"1", "2", "3", "4", "5"}
SCORE_FORMAT = "Return only one digit from 1 to 5."

if RUN_LIVE:
    base = f"Rate the quality of this {TEXT_KIND} from 1 (weak) to 5 (excellent)."
    scoring_prompts = {
        "own_judgment": f"{base} Use your own judgment. {SCORE_FORMAT}",
        "researcher_instructions": f"{base} {QUALITY_CRITERIA} {SCORE_FORMAT}",
    }
    scoring_jobs = (
        job for mode, instructions in scoring_prompts.items()
        for job in single_jobs(instructions, mode)
    )
    export_results(
        "scoring.csv", parse_choices(request_rows("scoring", scoring_jobs), SCORES, "score")
    )

## Experiment 4 · Generate scoring rubrics


Each selected model describes the five levels of a quality rubric for
this dataset. The rubric must contain a description for each score.

In [ ]:
def rubric_descriptions(text):
    try:
        rubric = json.loads(clean_answer(text))
    except ValueError:
        return {}
    if isinstance(rubric, dict) and set(rubric) == SCORES and all(
        isinstance(value, str) and value.strip() for value in rubric.values()
    ):
        return rubric
    return {}


def collect_rubrics(rubrics):
    instructions = (
        f"Create a quality rubric for evaluating a {TEXT_KIND}. {QUALITY_CRITERIA} "
        'Return a JSON object with keys "1", "2", "3", "4", "5". '
        "Each value must be a clear description of that score level."
    )
    jobs = (Job(alias, instructions, {}) for alias in MODELS)
    for row in request_rows("rubrics", jobs):
        descriptions = rubric_descriptions(row["raw_response"]) if row["status"] == "ok" else {}
        if descriptions:
            rubrics[row["model"]] = descriptions
        elif row["status"] == "ok":
            row["status"] = "invalid_answer"
        yield {**row, **{f"score_{grade}": descriptions.get(grade, "") for grade in sorted(SCORES)}}


if RUN_LIVE:
    rubrics = {}
    export_results("rubrics.csv", collect_rubrics(rubrics))

## Experiment 5 · Score texts with generated rubrics


Apply every valid generated rubric with every judge, including the
judge that created it. The output records both the judge and the rubric
author.

In [ ]:
def rubric_score_jobs():
    for author, descriptions in rubrics.items():
        scale = "\n".join(f"{grade}: {descriptions[grade]}" for grade in sorted(descriptions))
        instructions = f"Rate this {TEXT_KIND} using the following rubric.\n{scale}\n{SCORE_FORMAT}"
        for job in single_jobs(instructions, "model_rubric"):
            yield Job(job.model, job.prompt, {
                **job.labels, "rubric_author": author, "rubric_api_model": MODELS[author].api_id,
            })


if RUN_LIVE:
    export_results(
        "scoring_with_rubrics.csv",
        parse_choices(request_rows("scoring_with_rubrics", rubric_score_jobs()), SCORES, "score"),
        empty_fields=(
            "row_id", "source_column", "mode", "rubric_author", "rubric_api_model", "model",
            "api_model", "raw_response", "returned_model", "finish_reason", "status", "error", "score",
        ),
    )

## Experiment 6 · Pairwise quality preference


Compare the same pairs across five calls in the order normal, swapped,
normal, swapped, normal. Map each vote back to the source column.
Report a majority when a source receives at least three votes out of
five. At temperature zero, repeated votes serve as consistency checks.

In [ ]:
if RUN_LIVE:
    instructions = (
        f"Compare the quality of the two texts. Each is a {TEXT_KIND}. {QUALITY_CRITERIA} "
        'Which text is better? Return only "1" or "2".'
    )
    export_results(
        "preference.csv",
        pair_results("preference", instructions, ("N1", "S1", "N2", "S2", "N3")),
    )
    print(f"Run directory: {run_dir}")